GEETA 4B — parent/tutor portal store (local SQLite on this Mac).

Added 2026-10-13 at the project owner's request: sign-in for parents and
tutors, kid profiles, and a per-kid learning history used for reports and to
give the tutor a short memory of what the child has already learned.

**Privacy decision (owner, 2026-10-13) — amends PROJECT_BRIEF §6 "nothing saved":**
only a *learning summary* is kept per session (skills practised, solved/tried,
hint counts, mistake *categories*, grade, time) plus the parent PDF. No chat
text, no homework photos, no camera data. Passwords are argon2-hashed; sign-in
tokens are stored as SHA-256 hashes. A parent (or admin) can erase a child and
everything kept about them in one step (`delete_kid`).


## Bootstrap

This notebook is a GEETA runtime module. Executed standalone (e.g. `jupyter execute tools/verifier.ipynb`),
the next cell registers the notebook-as-module loader so `geeta.*` imports resolve to `tools/*.ipynb`.
When imported as `geeta.<name>` via `tools/_nbmodule.ipynb`, the same cell runs (idempotent).


In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501


In [ ]:

from __future__ import annotations

import base64
import re
import hashlib
import hmac
import json
import struct
import secrets
import shutil
import sqlite3
import threading
import time
from pathlib import Path
from typing import Any, Optional

from argon2 import PasswordHasher
from argon2.exceptions import InvalidHashError, VerificationError

_PH = PasswordHasher()
SESSION_DAYS = 14
DEVICE_DAYS = 90
KID_SESSION_HOURS = 12
AVATARS = ["🦊", "🐼", "🐯", "🦄", "🐸", "🐨", "🦁", "🐙", "🐧", "🚀", "🌈", "⚽"]

# additive migrations for stores created before a column existed
_MIGRATIONS = [
    ("kids", "pin_hash", "TEXT"),
    ("kids", "avatar", "TEXT"),
    ("kids", "world", "TEXT"),
    ("users", "totp_secret", "TEXT"),          # set once 2-step sign-in is switched on
    ("users", "totp_pending", "TEXT"),         # secret being set up, not active yet
    ("users", "totp_last_step", "INTEGER"),    # stops a code being used twice
    ("users", "recovery_hashes", "TEXT"),      # json list of sha256 of one-time codes
    ("kids", "study_code", "TEXT"),            # pilot: e.g. "P01" - the ONLY identifier in research exports
    ("users", "email_reports", "INTEGER"),     # unused since nightly emails were removed (28 Oct 2026)
]
EVENT_RETENTION_DAYS = 180
WORLDS = ["teddy", "car", "space", "dino", "unicorn", "ocean"]   # kid page themes (static/worlds.js)

SCHEMA = """
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY,
    email TEXT NOT NULL UNIQUE COLLATE NOCASE,
    name TEXT NOT NULL,
    role TEXT NOT NULL CHECK (role IN ('parent', 'tutor')),
    is_admin INTEGER NOT NULL DEFAULT 0,
    pw_hash TEXT NOT NULL,
    created REAL NOT NULL
);
CREATE TABLE IF NOT EXISTS auth_tokens (
    token_hash TEXT PRIMARY KEY,
    user_id INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
    expires REAL NOT NULL
);
CREATE TABLE IF NOT EXISTS kids (
    id INTEGER PRIMARY KEY,
    nickname TEXT NOT NULL,
    year_level INTEGER,
    parent_id INTEGER REFERENCES users(id) ON DELETE SET NULL,
    tutor_id INTEGER REFERENCES users(id) ON DELETE SET NULL,
    created REAL NOT NULL
);
-- one row per tutoring session: a LEARNING SUMMARY only. No chat text,
-- no homework images, no camera data (privacy decision 2026-10-13).
-- a home computer/tablet a parent or tutor has switched on for kid sign-in
CREATE TABLE IF NOT EXISTS device_tokens (
    token_hash TEXT PRIMARY KEY,
    user_id INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
    expires REAL NOT NULL
);
-- a kid signed in (picture + PIN) on such a device: tutoring only
CREATE TABLE IF NOT EXISTS kid_tokens (
    token_hash TEXT PRIMARY KEY,
    kid_id INTEGER NOT NULL REFERENCES kids(id) ON DELETE CASCADE,
    expires REAL NOT NULL
);
-- who did what, for the admin's security log (no passwords, codes or chat ever)
CREATE TABLE IF NOT EXISTS security_events (
    id INTEGER PRIMARY KEY,
    ts REAL NOT NULL,
    action TEXT NOT NULL,
    user_id INTEGER,
    kid_id INTEGER,
    detail TEXT,
    ip TEXT
);
CREATE INDEX IF NOT EXISTS security_events_ts ON security_events(ts);
-- password OK, waiting for the 2-step code (5 minutes)
CREATE TABLE IF NOT EXISTS pending_2fa (
    token_hash TEXT PRIMARY KEY,
    user_id INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
    expires REAL NOT NULL
);
-- pilot study: pre/post check scores entered by a tutor (research data uses study_code only)
CREATE TABLE IF NOT EXISTS pilot_scores (
    id INTEGER PRIMARY KEY,
    kid_id INTEGER NOT NULL REFERENCES kids(id) ON DELETE CASCADE,
    phase TEXT NOT NULL CHECK (phase IN ('pre', 'post')),
    score INTEGER NOT NULL,
    out_of INTEGER NOT NULL,
    ts REAL NOT NULL,
    entered_by INTEGER REFERENCES users(id) ON DELETE SET NULL,
    UNIQUE (kid_id, phase)
);
-- technical counts per saved session (numbers only - no text) for the admin report
CREATE TABLE IF NOT EXISTS session_tech (
    session_id INTEGER PRIMARY KEY REFERENCES learning_sessions(id) ON DELETE CASCADE,
    counts_json TEXT NOT NULL
);
-- which nightly reports went out, covering which period
CREATE TABLE IF NOT EXISTS report_log (
    id INTEGER PRIMARY KEY,
    kind TEXT NOT NULL,
    kid_id INTEGER REFERENCES kids(id) ON DELETE CASCADE,
    user_id INTEGER REFERENCES users(id) ON DELETE SET NULL,
    period_start REAL, period_end REAL, sent_ts REAL NOT NULL,
    status TEXT, detail TEXT
);
CREATE TABLE IF NOT EXISTS learning_sessions (
    id INTEGER PRIMARY KEY,
    kid_id INTEGER NOT NULL REFERENCES kids(id) ON DELETE CASCADE,
    started REAL NOT NULL,
    duration_s REAL,
    solved_count INTEGER,
    tried_count INTEGER,
    hints INTEGER,
    reengagements INTEGER,
    grade TEXT,
    attitude TEXT,
    confidence REAL,
    engagement REAL,
    focus_pct REAL,
    skills_json TEXT,     -- [{"skill", "tried", "solved", "hints"}]
    mistakes_json TEXT,   -- {"jump_size": 2, ...} category counts only
    pdf_name TEXT
);
"""


def _hash_token(tok: str) -> str:
    return hashlib.sha256(tok.encode()).hexdigest()


class PortalStore:
    """Local SQLite store for the parent/tutor portal (one file on this Mac)."""

    def __init__(self, db_path: str | Path, reports_dir: str | Path):
        self.db_path = Path(db_path)
        self.db_path.parent.mkdir(parents=True, exist_ok=True)
        self.reports_dir = Path(reports_dir)
        self.reports_dir.mkdir(parents=True, exist_ok=True)
        self._lock = threading.Lock()
        with self._conn() as c:
            c.executescript(SCHEMA)
            for table, col, typ in _MIGRATIONS:
                have = {r[1] for r in c.execute(f"PRAGMA table_info({table})")}
                if col not in have:
                    c.execute(f"ALTER TABLE {table} ADD COLUMN {col} {typ}")
        # children's learning data: readable by this Mac user account only
        for p in (self.db_path.parent, self.reports_dir):
            p.chmod(0o700)
        self.db_path.chmod(0o600)

    def _conn(self) -> sqlite3.Connection:
        c = sqlite3.connect(self.db_path, timeout=10)
        c.row_factory = sqlite3.Row
        c.execute("PRAGMA foreign_keys = ON")
        return c

    def _q(self, sql: str, args: tuple = (), one: bool = False):
        with self._lock, self._conn() as c:
            cur = c.execute(sql, args)
            rows = [dict(r) for r in cur.fetchall()]
            if sql.lstrip().upper().startswith("INSERT"):
                return cur.lastrowid
        return (rows[0] if rows else None) if one else rows

    # -- users ----------------------------------------------------------------
    def user_count(self) -> int:
        return self._q("SELECT COUNT(*) AS n FROM users", one=True)["n"]

    def create_user(self, email: str, name: str, role: str, password: str,
                    is_admin: bool = False) -> int:
        email = email.strip().lower()
        if role not in ("parent", "tutor"):
            raise ValueError("role must be parent or tutor")
        if "@" not in email or len(email) > 200:
            raise ValueError("please use a valid email address")
        check_password_strength(password)
        if self._q("SELECT id FROM users WHERE email = ?", (email,), one=True):
            raise ValueError("that email already has an account")
        return self._q("INSERT INTO users (email, name, role, is_admin, pw_hash, created) "
                       "VALUES (?, ?, ?, ?, ?, ?)",
                       (email, name.strip()[:80] or email, role, int(is_admin),
                        _PH.hash(password), time.time()))

    def verify_login(self, email: str, password: str) -> Optional[dict]:
        u = self._q("SELECT * FROM users WHERE email = ?", ((email or "").strip().lower(),), one=True)
        if not u:
            _PH.hash(password or "x")          # same work either way: no user-exists timing leak
            return None
        try:
            _PH.verify(u["pw_hash"], password or "")
        except (VerificationError, InvalidHashError):
            return None
        if _PH.check_needs_rehash(u["pw_hash"]):
            self._q("UPDATE users SET pw_hash = ? WHERE id = ?", (_PH.hash(password), u["id"]))
        return public_user(u)

    def set_password(self, user_id: int, password: str) -> None:
        check_password_strength(password)
        self._q("UPDATE users SET pw_hash = ? WHERE id = ?", (_PH.hash(password), user_id))
        self._q("DELETE FROM auth_tokens WHERE user_id = ?", (user_id,))   # sign out everywhere

    # -- security log ---------------------------------------------------------
    def log_event(self, action: str, user_id: int | None = None, kid_id: int | None = None,
                  detail: str = "", ip: str = "") -> None:
        try:
            self._q("INSERT INTO security_events (ts, action, user_id, kid_id, detail, ip) VALUES (?,?,?,?,?,?)",
                    (time.time(), action[:40], user_id, kid_id, (detail or "")[:200], (ip or "")[:64]))
            if secrets.randbelow(200) == 0:       # occasional tidy-up
                self._q("DELETE FROM security_events WHERE ts < ?",
                        (time.time() - EVENT_RETENTION_DAYS * 86400,))
        except Exception:
            pass                                    # logging must never break sign-in

    def recent_events(self, limit: int = 100) -> list[dict]:
        return self._q("SELECT e.*, u.name AS user_name, u.email AS user_email, k.nickname AS kid_name "
                       "FROM security_events e LEFT JOIN users u ON u.id = e.user_id "
                       "LEFT JOIN kids k ON k.id = e.kid_id ORDER BY e.ts DESC LIMIT ?", (limit,))

    # -- 2-step sign-in ---------------------------------------------------------
    def totp_start(self, user_id: int) -> str:
        secret = new_totp_secret()
        self._q("UPDATE users SET totp_pending = ? WHERE id = ?", (secret, user_id))
        return secret

    def totp_confirm(self, user_id: int, code: str) -> Optional[list[str]]:
        """Switch 2-step on if the code matches the pending secret; returns the
        one-time recovery codes (shown once, only their hashes are kept)."""
        row = self._q("SELECT totp_pending FROM users WHERE id = ?", (user_id,), one=True)
        if not row or not row["totp_pending"]:
            return None
        step = totp_match(row["totp_pending"], code)
        if step is None:
            return None
        codes = [f"{secrets.token_hex(2)}-{secrets.token_hex(2)}" for _ in range(8)]
        self._q("UPDATE users SET totp_secret = totp_pending, totp_pending = NULL, totp_last_step = ?, "
                "recovery_hashes = ? WHERE id = ?",
                (step, json.dumps([_hash_token(c) for c in codes]), user_id))
        return codes

    def totp_disable(self, user_id: int) -> None:
        self._q("UPDATE users SET totp_secret = NULL, totp_pending = NULL, totp_last_step = NULL, "
                "recovery_hashes = NULL WHERE id = ?", (user_id,))

    def totp_check(self, user_id: int, code: str) -> bool:
        """A current authenticator code (each usable once) or an unused recovery code."""
        row = self._q("SELECT totp_secret, totp_last_step, recovery_hashes FROM users WHERE id = ?",
                      (user_id,), one=True)
        if not row or not row["totp_secret"]:
            return False
        step = totp_match(row["totp_secret"], code)
        if step is not None:
            if row["totp_last_step"] is not None and step <= row["totp_last_step"]:
                return False                         # replayed code
            self._q("UPDATE users SET totp_last_step = ? WHERE id = ?", (step, user_id))
            return True
        rc = (code or "").strip().lower()
        hashes = json.loads(row["recovery_hashes"] or "[]")
        h = _hash_token(rc)
        if h in hashes:
            hashes.remove(h)
            self._q("UPDATE users SET recovery_hashes = ? WHERE id = ?", (json.dumps(hashes), user_id))
            return True
        return False

    def new_pending_2fa(self, user_id: int) -> str:
        tok = secrets.token_urlsafe(32)
        self._q("DELETE FROM pending_2fa WHERE expires < ?", (time.time(),))
        self._q("INSERT INTO pending_2fa (token_hash, user_id, expires) VALUES (?, ?, ?)",
                (_hash_token(tok), user_id, time.time() + 300))
        return tok

    def pending_2fa_user(self, tok: str | None, consume: bool = False) -> Optional[int]:
        if not tok:
            return None
        row = self._q("SELECT user_id FROM pending_2fa WHERE token_hash = ? AND expires > ?",
                      (_hash_token(tok), time.time()), one=True)
        if row and consume:
            self._q("DELETE FROM pending_2fa WHERE token_hash = ?", (_hash_token(tok),))
        return row["user_id"] if row else None

    def get_user(self, user_id: int) -> Optional[dict]:
        u = self._q("SELECT * FROM users WHERE id = ?", (user_id,), one=True)
        return public_user(u) if u else None

    def list_users(self) -> list[dict]:
        return [public_user(u) for u in self._q("SELECT * FROM users ORDER BY role, name")]

    def counts(self) -> dict:
        one = lambda sql: self._q(sql, one=True)["n"]
        return {"parents": one("SELECT COUNT(*) n FROM users WHERE role = 'parent'"),
                "tutors": one("SELECT COUNT(*) n FROM users WHERE role = 'tutor'"),
                "kids": one("SELECT COUNT(*) n FROM kids"),
                "sessions": one("SELECT COUNT(*) n FROM learning_sessions"),
                "sessions_7d": self._q("SELECT COUNT(*) n FROM learning_sessions WHERE started > ?",
                                       (time.time() - 7 * 86400,), one=True)["n"]}

    def delete_user(self, user_id: int) -> None:
        self._q("DELETE FROM users WHERE id = ?", (user_id,))

    # -- sign-in tokens -------------------------------------------------------
    def new_token(self, user_id: int) -> str:
        tok = secrets.token_urlsafe(32)
        self._q("DELETE FROM auth_tokens WHERE expires < ?", (time.time(),))
        self._q("INSERT INTO auth_tokens (token_hash, user_id, expires) VALUES (?, ?, ?)",
                (_hash_token(tok), user_id, time.time() + SESSION_DAYS * 86400))
        return tok

    def user_for_token(self, tok: str | None) -> Optional[dict]:
        if not tok:
            return None
        row = self._q("SELECT u.* FROM auth_tokens t JOIN users u ON u.id = t.user_id "
                      "WHERE t.token_hash = ? AND t.expires > ?",
                      (_hash_token(tok), time.time()), one=True)
        return public_user(row) if row else None

    def drop_token(self, tok: str | None) -> None:
        if tok:
            self._q("DELETE FROM auth_tokens WHERE token_hash = ?", (_hash_token(tok),))

    # -- kids -----------------------------------------------------------------
    def add_kid(self, nickname: str, year_level: int | None, parent_id: int | None,
                tutor_id: int | None) -> int:
        nickname = (nickname or "").strip()[:40]
        if not nickname:
            raise ValueError("please give the child a first name or nickname")
        if year_level is not None and year_level not in (0, 1, 2, 3, 4, 5, 6):
            raise ValueError("year level should be between Foundation (0) and 6")
        return self._q("INSERT INTO kids (nickname, year_level, parent_id, tutor_id, created) "
                       "VALUES (?, ?, ?, ?, ?)",
                       (nickname, year_level, parent_id, tutor_id, time.time()))

    def update_kid(self, kid_id: int, **fields) -> None:
        if "avatar" in fields and fields["avatar"] not in AVATARS + [None]:
            raise ValueError("please pick one of the pictures")
        if "world" in fields and fields["world"] not in WORLDS + [None]:
            raise ValueError("please pick one of the worlds")
        if fields.get("study_code") not in (None, "") and not re.fullmatch(r"[A-Z0-9-]{1,12}", fields["study_code"]):
            raise ValueError("study code: up to 12 capital letters, numbers or dashes (e.g. P01)")
        if fields.get("study_code") == "":
            fields["study_code"] = None
        allowed = {k: v for k, v in fields.items()
                   if k in ("nickname", "year_level", "parent_id", "tutor_id", "avatar", "world", "study_code")}
        for k, v in allowed.items():
            self._q(f"UPDATE kids SET {k} = ? WHERE id = ?", (v, kid_id))

    def get_kid(self, kid_id: int) -> Optional[dict]:
        k = self._q("SELECT * FROM kids WHERE id = ?", (kid_id,), one=True)
        if k:
            k["has_pin"] = k.pop("pin_hash") is not None
        return k

    def kids_for(self, user: dict) -> list[dict]:
        base = ("SELECT k.id, k.nickname, k.year_level, k.parent_id, k.tutor_id, k.created, "
                "k.avatar, k.world, k.study_code, (k.pin_hash IS NOT NULL) AS has_pin, "
                "p.name AS parent_name, t.name AS tutor_name, "
                "(SELECT COUNT(*) FROM learning_sessions s WHERE s.kid_id = k.id) AS n_sessions, "
                "(SELECT MAX(started) FROM learning_sessions s WHERE s.kid_id = k.id) AS last_session "
                "FROM kids k LEFT JOIN users p ON p.id = k.parent_id "
                "LEFT JOIN users t ON t.id = k.tutor_id ")
        if user.get("is_admin"):
            return self._q(base + "ORDER BY k.nickname")
        col = "parent_id" if user["role"] == "parent" else "tutor_id"
        return self._q(base + f"WHERE k.{col} = ? ORDER BY k.nickname", (user["id"],))

    # -- kid sign-in: a grown-up switches a device on, the kid taps picture + PIN --
    def set_kid_pin(self, kid_id: int, pin: str) -> None:
        if not (isinstance(pin, str) and pin.isdigit() and len(pin) == 4):
            raise ValueError("the PIN needs exactly 4 numbers")
        if pin in ("0000", "1111", "1234", "4321", "2222", "9999"):
            raise ValueError("that PIN is too easy to guess - try a different one")
        self._q("UPDATE kids SET pin_hash = ? WHERE id = ?", (_PH.hash(pin), kid_id))
        self._q("DELETE FROM kid_tokens WHERE kid_id = ?", (kid_id,))

    def verify_kid_pin(self, kid_id: int, pin: str) -> bool:
        row = self._q("SELECT pin_hash FROM kids WHERE id = ?", (kid_id,), one=True)
        if not row or not row["pin_hash"]:
            return False
        try:
            return _PH.verify(row["pin_hash"], pin or "")
        except (VerificationError, InvalidHashError):
            return False

    def new_device_token(self, user_id: int) -> str:
        tok = secrets.token_urlsafe(32)
        self._q("DELETE FROM device_tokens WHERE expires < ?", (time.time(),))
        self._q("INSERT INTO device_tokens (token_hash, user_id, expires) VALUES (?, ?, ?)",
                (_hash_token(tok), user_id, time.time() + DEVICE_DAYS * 86400))
        return tok

    def user_for_device(self, tok: str | None) -> Optional[dict]:
        if not tok:
            return None
        row = self._q("SELECT u.* FROM device_tokens d JOIN users u ON u.id = d.user_id "
                      "WHERE d.token_hash = ? AND d.expires > ?",
                      (_hash_token(tok), time.time()), one=True)
        return public_user(row) if row else None

    def drop_device_token(self, tok: str | None) -> None:
        if tok:
            self._q("DELETE FROM device_tokens WHERE token_hash = ?", (_hash_token(tok),))

    def new_kid_token(self, kid_id: int) -> str:
        tok = secrets.token_urlsafe(32)
        self._q("DELETE FROM kid_tokens WHERE expires < ?", (time.time(),))
        self._q("INSERT INTO kid_tokens (token_hash, kid_id, expires) VALUES (?, ?, ?)",
                (_hash_token(tok), kid_id, time.time() + KID_SESSION_HOURS * 3600))
        return tok

    def kid_for_token(self, tok: str | None) -> Optional[dict]:
        if not tok:
            return None
        row = self._q("SELECT k.id FROM kid_tokens t JOIN kids k ON k.id = t.kid_id "
                      "WHERE t.token_hash = ? AND t.expires > ?",
                      (_hash_token(tok), time.time()), one=True)
        return self.get_kid(row["id"]) if row else None

    def drop_kid_token(self, tok: str | None) -> None:
        if tok:
            self._q("DELETE FROM kid_tokens WHERE token_hash = ?", (_hash_token(tok),))

    # -- pilot study ----------------------------------------------------------
    def set_pilot_score(self, kid_id: int, phase: str, score: int, out_of: int, user_id: int) -> None:
        if phase not in ("pre", "post"):
            raise ValueError("phase must be pre or post")
        if not (0 < out_of <= 50 and 0 <= score <= out_of):
            raise ValueError("score must be between 0 and the number of questions")
        self._q("INSERT INTO pilot_scores (kid_id, phase, score, out_of, ts, entered_by) VALUES (?,?,?,?,?,?) "
                "ON CONFLICT(kid_id, phase) DO UPDATE SET score=excluded.score, out_of=excluded.out_of, "
                "ts=excluded.ts, entered_by=excluded.entered_by",
                (kid_id, phase, score, out_of, time.time(), user_id))

    def pilot_scores(self, kid_id: int) -> dict:
        return {r["phase"]: r for r in self._q("SELECT phase, score, out_of, ts FROM pilot_scores WHERE kid_id = ?", (kid_id,))}

    def research_export(self) -> list[dict]:
        """De-identified pilot data: study code only - no names, emails, IPs, free text."""
        rows = []
        for k in self._q("SELECT id, study_code, year_level FROM kids WHERE study_code IS NOT NULL ORDER BY study_code"):
            p = self.progress(k["id"])
            sc = self.pilot_scores(k["id"])
            pre, post = sc.get("pre"), sc.get("post")
            pct = lambda r: round(100 * r["score"] / r["out_of"], 1) if r else None
            rows.append({
                "study_code": k["study_code"], "year_level": k["year_level"],
                "sessions": p["sessions"], "minutes": p["minutes"], "questions_solved": p["solved"],
                "skills_practised": len(p["skills"]),
                "skills_confident": sum(1 for s in p["skills"] if s["status"] == "confident"),
                "slip_categories": ";".join(f"{c}:{n}" for c, n in sorted(p["mistakes"].items())),
                "pre_score": pre["score"] if pre else None, "pre_out_of": pre["out_of"] if pre else None,
                "post_score": post["score"] if post else None, "post_out_of": post["out_of"] if post else None,
                "gain_pp": (round(pct(post) - pct(pre), 1) if pre and post else None),
            })
        return rows

    def all_kids(self) -> list[dict]:
        return [self.get_kid(r["id"]) for r in self._q("SELECT id FROM kids ORDER BY id")]

    def sessions_between(self, kid_id: int, start: float, end: float) -> list[dict]:
        return [s for s in self.sessions_for(kid_id) if start <= s["started"] < end or
                start <= (s["started"] + (s["duration_s"] or 0)) < end]

    def save_session_tech(self, session_id: int, counts: dict) -> None:
        self._q("INSERT OR REPLACE INTO session_tech (session_id, counts_json) VALUES (?, ?)",
                (session_id, json.dumps(counts)))

    def tech_between(self, start: float, end: float) -> dict:
        keys = ("exact_items", "model_items", "gate_blocked", "safe_fallbacks", "leak_blocks",
                "verifier_failed", "photos", "wrong_answer_risk")
        out = dict.fromkeys(keys, 0)
        for r in self._q("SELECT t.counts_json FROM session_tech t JOIN learning_sessions s ON s.id = t.session_id "
                         "WHERE s.started >= ? AND s.started < ?", (start, end)):
            for k, v in json.loads(r["counts_json"]).items():
                if k in out:
                    out[k] += int(v or 0)
        return out

    def usage_between(self, start: float, end: float) -> dict:
        rows = self._q("SELECT kid_id, duration_s, solved_count, tried_count, hints FROM learning_sessions "
                       "WHERE started >= ? AND started < ?", (start, end))
        c = self.counts()
        return {"sessions": len(rows), "kids_active": len({r["kid_id"] for r in rows}),
                "minutes": round(sum(r["duration_s"] or 0 for r in rows) / 60),
                "solved": sum(r["solved_count"] or 0 for r in rows), "tried": sum(r["tried_count"] or 0 for r in rows),
                "hints": sum(r["hints"] or 0 for r in rows),
                "parents": c["parents"], "tutors": c["tutors"], "kids_total": c["kids"]}

    def security_between(self, start: float, end: float) -> dict:
        return {r["action"]: r["n"] for r in self._q(
            "SELECT action, COUNT(*) AS n FROM security_events WHERE ts >= ? AND ts < ? GROUP BY action "
            "ORDER BY n DESC", (start, end))}

    def can_see_kid(self, user: dict, kid_id: int) -> bool:
        k = self.get_kid(kid_id)
        if not k:
            return False
        return bool(user.get("is_admin")) or k["parent_id"] == user["id"] or k["tutor_id"] == user["id"]

    def delete_kid(self, kid_id: int) -> None:
        """Erase a child and EVERYTHING kept about them (APP 11 / parent request)."""
        self._q("DELETE FROM kids WHERE id = ?", (kid_id,))
        shutil.rmtree(self.kid_report_dir(kid_id), ignore_errors=True)

    def kid_report_dir(self, kid_id: int) -> Path:
        return self.reports_dir / f"kid_{int(kid_id)}"

    # -- learning summaries ---------------------------------------------------
    def save_session(self, kid_id: int, summary: dict, pdf_name: str | None) -> int:
        return self._q(
            "INSERT INTO learning_sessions (kid_id, started, duration_s, solved_count, tried_count, "
            "hints, reengagements, grade, attitude, confidence, engagement, focus_pct, "
            "skills_json, mistakes_json, pdf_name) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
            (kid_id, summary["started"], summary["duration_s"], summary["solved_count"],
             summary["tried_count"], summary["hints"], summary["reengagements"],
             summary.get("grade"), summary.get("attitude"), summary.get("confidence"),
             summary.get("engagement"), summary.get("focus_pct"),
             json.dumps(summary["skills"]), json.dumps(summary["mistakes"]), pdf_name))

    def set_session_pdf(self, session_id: int, pdf_name: str) -> None:
        self._q("UPDATE learning_sessions SET pdf_name = ? WHERE id = ?", (pdf_name, session_id))

    def sessions_for(self, kid_id: int) -> list[dict]:
        rows = self._q("SELECT * FROM learning_sessions WHERE kid_id = ? ORDER BY started DESC",
                       (kid_id,))
        for r in rows:
            r["skills"] = json.loads(r.pop("skills_json") or "[]")
            r["mistakes"] = json.loads(r.pop("mistakes_json") or "{}")
        return rows

    def get_session(self, session_id: int) -> Optional[dict]:
        return self._q("SELECT * FROM learning_sessions WHERE id = ?", (session_id,), one=True)

    def progress(self, kid_id: int) -> dict:
        """Everything the portal + the tutor need about a child's learning so far."""
        sessions = self.sessions_for(kid_id)
        skills: dict[str, dict] = {}
        mistakes: dict[str, int] = {}
        for s in reversed(sessions):            # oldest first
            for sk in s["skills"]:
                d = skills.setdefault(sk["skill"], {"skill": sk["skill"], "tried": 0, "solved": 0,
                                                    "hints": 0, "slips": 0, "sessions": 0,
                                                    "first": s["started"], "last": s["started"]})
                d["tried"] += sk.get("tried", 0)
                d["solved"] += sk.get("solved", 0)
                d["hints"] += sk.get("hints", 0)
                d["slips"] += sk.get("slips", 0)
                d["sessions"] += 1
                d["last"] = s["started"]
            for cat, n in s["mistakes"].items():
                mistakes[cat] = mistakes.get(cat, 0) + n
        for d in skills.values():
            rate = d["solved"] / d["tried"] if d["tried"] else 0
            hints_per = (d["hints"] + d["slips"]) / d["tried"] if d["tried"] else 0
            d["status"] = ("confident" if rate >= 0.8 and hints_per <= 1.0 and d["solved"] >= 2
                           else "getting there" if rate >= 0.5 else "still learning")
        # recent mistakes weigh more: only last 5 sessions decide "watch for"
        recent: dict[str, int] = {}
        for s in sessions[:5]:
            for cat, n in s["mistakes"].items():
                recent[cat] = recent.get(cat, 0) + n
        try:
            from geeta.worksheet import MISTAKE_QUIET
        except Exception:
            MISTAKE_QUIET = set()
        watch = [c for c, n in sorted(recent.items(), key=lambda kv: -kv[1]) if n >= 2 and c not in MISTAKE_QUIET][:3]
        return {"sessions": len(sessions),
                "minutes": round(sum((s["duration_s"] or 0) for s in sessions) / 60),
                "solved": sum(s["solved_count"] or 0 for s in sessions),
                "skills": sorted(skills.values(), key=lambda d: -d["last"]),
                "mistakes": mistakes, "watch_for": watch,
                "last_session": sessions[0]["started"] if sessions else None}

    def profile_for_tutor(self, kid_id: int) -> dict:
        """A few lines for the tutor model - skills + mistake categories only."""
        from geeta.worksheet import MISTAKE_TEXT
        kid = self.get_kid(kid_id) or {}
        p = self.progress(kid_id)
        lines = []
        if kid.get("year_level") is not None:
            lines.append(f"Year {kid['year_level']}." if kid["year_level"] else "Foundation year.")
        if not p["sessions"]:
            lines.append("First session with GEETA.")
        else:
            lines.append(f"{p['sessions']} earlier session(s).")
            conf = [s["skill"] for s in p["skills"] if s["status"] == "confident"][:4]
            learn = [s["skill"] for s in p["skills"] if s["status"] != "confident"][:4]
            if conf:
                lines.append("Confident with: " + "; ".join(conf) + ".")
            if learn:
                lines.append("Still practising: " + "; ".join(learn) + ".")
            if p["watch_for"]:
                lines.append("Often trips on: " + "; ".join(MISTAKE_TEXT.get(c, c) for c in p["watch_for"]) + ".")
            if p["skills"]:
                lines.append(f"Last time: {p['skills'][0]['skill']}.")
        return {"text": " ".join(lines), "watch_for": p["watch_for"],
                "year": kid.get("year_level"),
                "last_skill": p["skills"][0]["skill"] if p["skills"] else None,
                "sessions": p["sessions"]}


def public_user(u: dict) -> dict:
    return {"id": u["id"], "email": u["email"], "name": u["name"], "role": u["role"],
            "is_admin": bool(u["is_admin"]), "two_step": bool(u.get("totp_secret"))}


# ---- TOTP (RFC 6238) - works with Google/Microsoft Authenticator, 1Password, etc.
def new_totp_secret() -> str:
    return base64.b32encode(secrets.token_bytes(20)).decode().rstrip("=")


def totp_at(secret: str, step: int) -> str:
    key = base64.b32decode(secret + "=" * (-len(secret) % 8))
    h = hmac.new(key, struct.pack(">Q", step), hashlib.sha1).digest()
    o = h[-1] & 0x0F
    return f"{(struct.unpack('>I', h[o:o + 4])[0] & 0x7FFFFFFF) % 1_000_000:06d}"


def totp_match(secret: str, code: str, now: float | None = None) -> Optional[int]:
    """The 30-second step the code belongs to (allowing +/- one step of clock drift)."""
    code = "".join(ch for ch in (code or "") if ch.isdigit())
    if len(code) != 6:
        return None
    step = int((now or time.time()) // 30)
    for s in (step - 1, step, step + 1):
        if hmac.compare_digest(totp_at(secret, s), code):
            return s
    return None


def check_password_strength(pw: str) -> None:
    if not pw or len(pw) < 10:
        raise ValueError("password needs at least 10 characters")
    if pw.lower() == pw or pw.upper() == pw or not any(ch.isdigit() for ch in pw):
        raise ValueError("password needs upper- and lower-case letters and a number")


def summarise_session(closed: dict, metrics: dict, grade: dict | None) -> dict:
    """TutorSession.close() -> the learning summary we keep. Deliberately drops
    every free-text field (kid messages, OCR text, hints) - only counts,
    skill names and mistake categories survive."""
    items = closed.get("items") or []
    events = closed.get("events") or []
    skills: dict[str, dict] = {}
    mistakes: dict[str, int] = {}
    if items:
        for it in items:
            d = skills.setdefault(it["skill"], {"skill": it["skill"], "tried": 0, "solved": 0,
                                                "hints": 0, "slips": 0})
            d["tried"] += 1
            d["slips"] += len(it.get("mistakes") or [])
            d["solved"] += int(bool(it.get("solved")))
            d["hints"] += int(it.get("hints") or 0)
            for cat in it.get("mistakes") or []:
                mistakes[cat] = mistakes.get(cat, 0) + 1
    elif closed.get("skill_id") or closed.get("ocr_confirmed"):
        name = (closed.get("skill_id") or "homework question").replace("_", " ").capitalize()
        skills[name] = {"skill": name, "tried": 1, "solved": int(bool(closed.get("solved"))),
                        "hints": int(metrics.get("hints_used") or 0)}
    focus = closed.get("focus") or {}
    t0 = min((e.get("t") for e in events if isinstance(e.get("t"), (int, float))), default=time.time())
    return {
        "started": t0,
        "duration_s": float(closed.get("duration_s") or 0),
        "solved_count": sum(d["solved"] for d in skills.values()),
        "tried_count": sum(d["tried"] for d in skills.values()),
        "hints": int(metrics.get("hints_used") or 0),
        "reengagements": int(metrics.get("reengagements") or 0),
        "grade": (grade or {}).get("letter"),
        "attitude": (grade or {}).get("attitude"),
        "confidence": metrics.get("confidence"),
        "engagement": metrics.get("engagement"),
        "focus_pct": focus.get("engaged_pct"),
        "skills": list(skills.values()),
        "mistakes": mistakes,
    }
